# DESC ELAsTiCC2 — 04 : Exploration visuelle des features Bazin — SNIa vs non-Ia

Ce notebook charge directement les fichiers parquet bruts produits par
`03_elasticc2_fit_bazin_extract_features.ipynb` (toutes les classes, **avant**
filtrage qualité strict et **avant** standardisation), construit le label binaire
SNIa / non-Ia, et compare visuellement la distribution de chaque feature entre
les deux groupes :

- **histogrammes superposés** (un par feature, grille de subplots),
- **boxplots côte à côte** (mêmes grilles, vue complémentaire des médianes/quartiles).

Objectif : identifier à l'œil les features qui séparent déjà bien SNIa et non-Ia
avant même d'entraîner un classifieur, et repérer d'éventuelles anomalies (valeurs
aberrantes, distributions très asymétriques) qui pourraient justifier une
transformation avant standardisation (notebook 06).

- author : Sylvie Dagoret-Campagne
- creation date : 2026-06-20
- input : `features/bazin_features_<obj_class>.parquet` produits par `03_elasticc2_fit_bazin_extract_features.ipynb`
- note : ce notebook se place **avant** le split train/test (`05_build_train_test_sets.ipynb`) —
 il explore l'ensemble des objets disponibles, pas seulement l'échantillon d'entraînement.

## 0 · Imports

In [ ]:
%matplotlib inline

import sys
import os
import pathlib
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_logger = logging.getLogger("main")
if not _logger.hasHandlers():
    _logout = logging.StreamHandler(sys.stderr)
    _logger.addHandler(_logout)
    _logout.setFormatter(logging.Formatter(
        '[%(asctime)s - %(levelname)s] - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    ))
_logger.setLevel(logging.INFO)
_logger.info("Imports done.")

## 1 · Paramètres

- `FEATURES_DIR` : dossier contenant les fichiers parquet produits par le notebook 03.
- `OBJ_CLASS_LIST` : classes à inclure.
- `REQUIRE_GOOD_FIT` : si `True`, ne garde que les lignes `is_good_fit == True` avant de tracer
  (recommandé pour une comparaison propre ; mettre `False` pour voir l'effet du filtrage qualité).
- `N_COLS` : nombre de colonnes dans les grilles de subplots.
- `BINS` : nombre de bins pour les histogrammes.

In [ ]:
# ── Paramètres principaux ────────────────────────────────────────────────────

FEATURES_DIR = pathlib.Path(os.getcwd()) / "features"

OBJ_CLASS_LIST = [
    'SNIa-SALT3',
    'SNIb-Templates',
    'SNIc-Templates',
    'SNII-Templateset',
]

REQUIRE_GOOD_FIT = True   # ne garder que is_good_fit == True avant de tracer
N_COLS = 6                # colonnes par grille de subplots
BINS   = 25               # nombre de bins pour les histogrammes

print(f"FEATURES_DIR = {FEATURES_DIR}")

## 2 · Chargement, filtrage qualité, construction du label

In [ ]:
def load_all_features(features_dir, obj_class_list):
    """
    Charge un parquet par classe (bazin_features_<obj_class>.parquet) et les
    concatène en un seul DataFrame.
    """
    dfs = []
    for obj_class in obj_class_list:
        fpath = features_dir / f"bazin_features_{obj_class}.parquet"
        if not fpath.is_file():
            _logger.warning(f"Fichier manquant pour {obj_class} : {fpath} — classe ignorée.")
            continue
        df = pd.read_parquet(fpath)
        _logger.info(f"[{obj_class}] {len(df)} lignes chargées depuis {fpath.name}")
        dfs.append(df)

    if not dfs:
        raise FileNotFoundError(
            f"Aucun fichier de features trouvé dans {features_dir}. "
            f"Avez-vous exécuté 03_elasticc2_fit_bazin_extract_features.ipynb ?"
        )
    return pd.concat(dfs, ignore_index=True)


df_all = load_all_features(FEATURES_DIR, OBJ_CLASS_LIST)
print(f"Total chargé : {len(df_all)} objets, {df_all['obj_class'].nunique()} classes.")
print(df_all.groupby('obj_class').size().rename('n_objects'))

In [ ]:
# ── Filtrage qualité ──────────────────────────────────────────────────────
if REQUIRE_GOOD_FIT:
    n_before = len(df_all)
    df = df_all[df_all['is_good_fit'] == True].copy()
    print(f"Filtrage is_good_fit==True : {len(df)}/{n_before} lignes conservées.")
else:
    df = df_all[df_all['fit_success'] == True].copy()
    print(f"Filtrage fit_success==True (is_good_fit non appliqué) : {len(df)}/{len(df_all)} lignes conservées.")

# ── Label binaire SNIa vs non-Ia ─────────────────────────────────────────
df['label'] = (df['obj_class'] == 'SNIa-SALT3').astype(int)
df['label_name'] = df['label'].map({1: 'SNIa', 0: 'non-Ia'})

print("\nRépartition du label :")
print(df.groupby(['label_name', 'obj_class']).size().rename('n_objects'))

## 3 · Définition des groupes de features

Pour des grilles lisibles, les features sont regroupées par nature :

- **Bazin par bande** : `{band}_A, t0, t_fall, t_rise, B, t_max, f_max, m_p` (8 par bande × 6 bandes = 48),
- **Couleurs au pic** : `c_ug, c_gr, c_ri, c_iz, c_zY` (5),
- **Redshift** : `redshift` (1),
- **Qualité du fit** (à titre indicatif, pas des features de classification au sens strict,
  mais utiles pour vérifier qu'elles ne diffèrent pas trop entre les deux populations) :
  `chi2_red`, `chi2_total`, `t_max_global`, `F_peak_global`.

In [ ]:
BANDS = ['u', 'g', 'r', 'i', 'z', 'Y']

bazin_cols = []
for band in BANDS:
    bazin_cols += [f'{band}_A', f'{band}_t0', f'{band}_t_fall', f'{band}_t_rise',
                   f'{band}_B', f'{band}_t_max', f'{band}_f_max', f'{band}_m_p']
color_cols = [f'c_{BANDS[i]}{BANDS[i+1]}' for i in range(len(BANDS) - 1)]
redshift_cols = ['redshift']
quality_cols = ['chi2_red', 'chi2_total', 't_max_global', 'F_peak_global']

feature_groups = {
    'Paramètres Bazin par bande': [c for c in bazin_cols if c in df.columns],
    'Couleurs au pic': [c for c in color_cols if c in df.columns],
    'Redshift': [c for c in redshift_cols if c in df.columns],
    'Qualité du fit (indicatif)': [c for c in quality_cols if c in df.columns],
}

for name, cols in feature_groups.items():
    print(f"{name:32s} : {len(cols)} colonnes")

## 4 · Fonctions de tracé : histogrammes et boxplots en grille

Les deux fonctions ignorent silencieusement les valeurs `NaN` (bandes non ajustées
pour un objet donné — c'est attendu et normal, surtout pour les bandes `u`/`Y`,
souvent moins bien échantillonnées).

In [ ]:
def plot_histograms_grid(df, feature_cols, label_col='label_name', n_cols=N_COLS,
                          bins=BINS, title=None):
    """
    Grille d'histogrammes superposés (un par feature), un groupe (SNIa / non-Ia)
    par couleur, densité normalisée pour rendre les populations comparables
    malgré leurs effectifs différents.
    """
    n_feat = len(feature_cols)
    if n_feat == 0:
        print("Aucune feature à tracer.")
        return None

    n_rows = int(np.ceil(n_feat / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 2.4 * n_rows), tight_layout=True)
    axes = np.atleast_1d(axes).ravel()

    groups = df.groupby(label_col)
    for i, col in enumerate(feature_cols):
        ax = axes[i]
        for name, sub in groups:
            vals = sub[col].dropna()
            if len(vals) == 0:
                continue
            ax.hist(vals, bins=bins, alpha=0.55, density=True, label=name)
        ax.set_title(col, fontsize=8)
        ax.tick_params(labelsize=6)

    for j in range(n_feat, len(axes)):
        axes[j].axis('off')

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='upper right', fontsize=9)
    if title:
        fig.suptitle(title, y=1.02, fontsize=12)
    return fig

In [ ]:
def plot_boxplots_grid(df, feature_cols, label_col='label_name', n_cols=N_COLS, title=None):
    """
    Grille de boxplots côte à côte (SNIa vs non-Ia) pour chaque feature.
    Les outliers extrêmes sont masqués (`showfliers=False`) pour ne pas écraser
    l'échelle ; utiliser `plot_histograms_grid` pour voir la distribution complète.
    """
    n_feat = len(feature_cols)
    if n_feat == 0:
        print("Aucune feature à tracer.")
        return None

    n_rows = int(np.ceil(n_feat / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 2.4 * n_rows), tight_layout=True)
    axes = np.atleast_1d(axes).ravel()

    label_order = sorted(df[label_col].unique())
    for i, col in enumerate(feature_cols):
        ax = axes[i]
        data = [df.loc[df[label_col] == lab, col].dropna().values for lab in label_order]
        data = [d if len(d) > 0 else np.array([np.nan]) for d in data]
        ax.boxplot(data, tick_labels=label_order, showfliers=False)
        ax.set_title(col, fontsize=8)
        ax.tick_params(labelsize=6)

    for j in range(n_feat, len(axes)):
        axes[j].axis('off')

    if title:
        fig.suptitle(title, y=1.02, fontsize=12)
    return fig

## 5 · Paramètres Bazin par bande — Histogrammes

48 features (8 paramètres × 6 bandes), affichées en une grille.

In [ ]:
fig = plot_histograms_grid(
    df, feature_groups['Paramètres Bazin par bande'],
    title="Paramètres Bazin par bande — SNIa vs non-Ia (histogrammes)"
)
plt.show()

## 6 · Paramètres Bazin par bande — Boxplots

In [ ]:
fig = plot_boxplots_grid(
    df, feature_groups['Paramètres Bazin par bande'],
    title="Paramètres Bazin par bande — SNIa vs non-Ia (boxplots)"
)
plt.show()

## 7 · Couleurs au pic — Histogrammes et boxplots

In [ ]:
fig = plot_histograms_grid(
    df, feature_groups['Couleurs au pic'], n_cols=5,
    title="Couleurs au pic — SNIa vs non-Ia (histogrammes)"
)
plt.show()

In [ ]:
fig = plot_boxplots_grid(
    df, feature_groups['Couleurs au pic'], n_cols=5,
    title="Couleurs au pic — SNIa vs non-Ia (boxplots)"
)
plt.show()

## 8 · Redshift et indicateurs de qualité du fit

In [ ]:
redshift_and_quality = feature_groups['Redshift'] + feature_groups['Qualité du fit (indicatif)']

fig = plot_histograms_grid(
    df, redshift_and_quality, n_cols=len(redshift_and_quality),
    title="Redshift et qualité du fit — SNIa vs non-Ia (histogrammes)"
)
plt.show()

In [ ]:
fig = plot_boxplots_grid(
    df, redshift_and_quality, n_cols=len(redshift_and_quality),
    title="Redshift et qualité du fit — SNIa vs non-Ia (boxplots)"
)
plt.show()

## 9 · Tableau récapitulatif : moyenne / écart-type / médiane par feature et par label

Utile pour repérer rapidement, en chiffres, les features qui séparent le mieux
les deux populations (grand écart de moyenne/médiane relativement à l'écart-type).

In [ ]:
all_feature_cols = (
    feature_groups['Paramètres Bazin par bande'] +
    feature_groups['Couleurs au pic'] +
    feature_groups['Redshift']
)

summary_stats = df.groupby('label_name')[all_feature_cols].agg(['mean', 'std', 'median'])
summary_stats

In [ ]:
# ── Score de séparation simple : |moyenne(SNIa) - moyenne(non-Ia)| / écart-type combiné ──
# (analogue à un d de Cohen, sans en avoir toutes les hypothèses — indicatif seulement.)

means = df.groupby('label_name')[all_feature_cols].mean()
stds = df.groupby('label_name')[all_feature_cols].std()

pooled_std = np.sqrt((stds.loc['SNIa']**2 + stds.loc['non-Ia']**2) / 2)
separation = ((means.loc['SNIa'] - means.loc['non-Ia']).abs() / pooled_std).sort_values(ascending=False)

print("Top 15 features les plus séparatrices (|écart de moyenne| / écart-type combiné) :")
separation.head(15).rename('separation_score').to_frame()

## 10 · Zoom sur les features les plus séparatrices

Histogrammes et boxplots détaillés pour les features identifiées comme les plus
discriminantes à la section précédente — utile pour confirmer visuellement le score
et repérer une éventuelle séparation non-linéaire ou bimodale que le score résume mal.

In [ ]:
TOP_N_ZOOM = 12
top_features = separation.head(TOP_N_ZOOM).index.tolist()

fig = plot_histograms_grid(
    df, top_features, n_cols=4,
    title=f"Top {TOP_N_ZOOM} features les plus séparatrices — Histogrammes"
)
plt.show()

In [ ]:
fig = plot_boxplots_grid(
    df, top_features, n_cols=4,
    title=f"Top {TOP_N_ZOOM} features les plus séparatrices — Boxplots"
)
plt.show()

## 11 · Notes

- Ces distributions sont tracées **avant standardisation** (`StandardScaler`), contrairement
 aux features utilisées dans `06_train_classifier.ipynb` — l'échelle brute (flux en compte ADU,
 temps en jours, magnitudes) est donc directement interprétable physiquement ici.
- Le score de séparation de la Section 9 est purement indicatif (différence de moyennes
 normalisée) : il ne capture pas les séparations non-linéaires que Random Forest / XGBoost
 peuvent exploiter — voir l'analyse SHAP du notebook 06 pour une vue complémentaire, fondée
 sur le modèle entraîné plutôt que sur les seules statistiques marginales.
- Si `REQUIRE_GOOD_FIT=False` fait apparaître une population substantiellement différente,
 cela peut indiquer un biais de sélection du flag `is_good_fit` entre les classes (par exemple
 si une classe a des courbes de lumière intrinsèquement plus bruitées) — à garder en tête
 pour l'interprétation des performances du classifieur.